In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path

import torch
import torch.multiprocessing as mp
from jppype import vscode_theme
from torch_geometric.loader import DataLoader
from torch_geometric.transforms import ToDevice

from fundus_toolkits import FundusData
from fundus_vessels_toolkit.models.topology.dataset import BranchDigraphDataset

vscode_theme()
# mp.set_start_method("spawn", force=True)

HTML(value="<style>\n        .cell-output-ipywidget-background {\n                background: transparent !imp…

In [ ]:
PATH = [
    Path("/run/media/gaby/GREY SSD/PostDoc/DATA/Fundus/" + folder)
    for folder in ["GAVE-train", "MAPLES-DR", "Fundus-AV", "LES-AV", "INSPIRE", "AV_DRIVE/training"]
]
RAW = [path / "1-images" for path in PATH]
AV = [path / "2-av-pred_CLEMENT" for path in PATH]
TOPO = [path / "3-topo" for path in PATH]

dataset = BranchDigraphDataset.load_from_dirs(RAW, TOPO, AV, resize_to=1024, output_dir="tmp/dataset-test")

In [3]:
dataset = BranchDigraphDataset("ALL_DATA_bundle.tar.gz")

Processing...
Done!


In [ ]:
len(dataset)

## Graph Augment


In [ ]:
ID = 0

In [ ]:
m, digraph, _ = dataset.jppype_show(ID, augment=True)
m

In [ ]:
dataset.get(20, augment=True, version="fvt")
%timeit dataset.get(20, augment=True, version='fvt')

222 ms ± 6.77 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [17]:
import cProfile

dataset.preload()

Preloading dataset: 100%|██████████| 373/373 [00:33<00:00, 11.12it/s]


BranchDigraphDataset(373)

In [26]:
cProfile.run("dataset.get(20, augment=True, version='fvt')", sort="cumulative")

         472740 function calls (468334 primitive calls) in 0.343 seconds

   Ordered by: cumulative time

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
      5/3    0.000    0.000    0.399    0.133 queue.py:180(get)
        8    0.000    0.000    0.397    0.050 threading.py:327(wait)
      3/1    0.000    0.000    0.343    0.343 {built-in method builtins.exec}
        4    0.000    0.000    0.200    0.050 threading.py:641(wait)
        2    0.000    0.000    0.078    0.039 tree_topology.py:248(read_branch_topo)
        2    0.017    0.009    0.078    0.039 tree_topology.py:513(read_branch_topology)
        1    0.004    0.004    0.067    0.067 vbranch_digraph.py:1279(prepare_graph_for_reconnections)
        1    0.000    0.000    0.052    0.052 vgraph.py:2154(transform)
        1    0.007    0.007    0.052    0.052 vgeometric_data.py:2265(transform)
      116    0.003    0.000    0.047    0.000 vgraph.py:2768(split_branch)
        1    0.000    0.000    0.030 

BranchDigraphData(edge_index=[2, 17984], pos=[381, 2], edge_dir=[17984, 2], branch_nodes=[381, 2], branch_curves=[381, 20, 2], branch_root_candidates=[381, 2], branch_tip_pos=[381, 2, 2], branch_tip_tan=[381, 2, 2], edge_p=[17984], branch_root_p=[381, 2], branch_fp_p=[381], branch_av_p=[381], branch_dir_p=[381], branch_subtree_idx=[381], img=[3, 1024, 1024], od_yx=[2], mac_yx=[2], vnode_count=393, vnode_coord=[393, 2], name='01_g/fvt', num_nodes=381)

In [ ]:
from fundus_vessels_toolkit.utils.nnet.profiling import Profiler

Profiler.reset()
dataset.get(20, augment=True, version="fvt")
print(Profiler.get("split_branch").print())

split_branch:                  16.63ms (runs=105, avg= 158.4µs)
├── split geometric data:       3.35ms (runs=105, avg=  31.9µs)
└── update graph structure:    12.54ms (runs=105, avg= 119.4µs)
